# Trefoil Knot 

In [1]:
# Imports and Setup
import numpy as np # type: ignore
import matplotlib.pyplot as plt # type: ignore
from matplotlib.animation import FuncAnimation # type: ignore
import os

# Create the directory if it doesn't exist
output_dir = "output/trefoil_animation"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Torus Knot Definitions
# Tube parameters
r = 0.2  # Tube radius
t_vals = np.linspace(0, 2 * np.pi, 500)
theta_vals = np.linspace(0, 2 * np.pi, 30)

def torus_trefoil_knot(t):
    x = (2 + np.cos(3 * t)) * np.cos(2 * t)
    y = (2 + np.cos(3 * t)) * np.sin(2 * t)
    z = np.sin(3 * t)
    return x, y, z

def torus_trefoil_tangent(t):
    dx = -2 * (2 + np.cos(3 * t)) * np.sin(2 * t) - 3 * np.sin(3 * t) * np.cos(2 * t)
    dy =  2 * (2 + np.cos(3 * t)) * np.cos(2 * t) - 3 * np.sin(3 * t) * np.sin(2 * t)
    dz =  3 * np.cos(3 * t)
    return dx, dy, dz

def normalize(x, y, z):
    norm = np.sqrt(x ** 2 + y ** 2 + z ** 2)
    return x / norm, y / norm, z / norm

def build_tube():
    xk, yk, zk = torus_trefoil_knot(t_vals)
    dx, dy, dz = torus_trefoil_tangent(t_vals)
    tx, ty, tz = normalize(dx, dy, dz)
    bx, by, bz = normalize(-ty, tx, 0)
    nx, ny, nz = np.cross([tx, ty, tz], [bx, by, bz], axis=0)
    T, Theta = np.meshgrid(t_vals, theta_vals, indexing='ij')
    X = xk[:, None] + r * (bx[:, None] * np.cos(Theta) + nx[:, None] * np.sin(Theta))
    Y = yk[:, None] + r * (by[:, None] * np.cos(Theta) + ny[:, None] * np.sin(Theta))
    Z = zk[:, None] + r * (bz[:, None] * np.cos(Theta) + nz[:, None] * np.sin(Theta))
    return X, Y, Z

def rotate_y(X, Y, Z, angle_deg):
    angle_rad = np.radians(angle_deg)
    cos_a, sin_a = np.cos(angle_rad), np.sin(angle_rad)
    X_rot = X * cos_a + Z * sin_a
    Z_rot = -X * sin_a + Z * cos_a
    return X_rot, Y, Z_rot


In [ ]:
# Animation and Save
X_base, Y_base, Z_base = build_tube()

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(projection='3d')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_zlim(-2, 2)
ax.axis('off')
surf = [ax.plot_surface(X_base, Y_base, Z_base, color=(0, 0.25, 1), linewidth=0, antialiased=True)]

def update(frame):
    angle = frame
    if surf[0] in ax.collections:
        surf[0].remove()
    Xr, Yr, Zr = rotate_y(X_base, Y_base, Z_base, angle)
    surf[0] = ax.plot_surface(Xr, Yr, Zr, color=(0, 0.25, 1), linewidth=0, antialiased=True)
    return surf

anim = FuncAnimation(fig, update, frames=360, interval=30, blit=False)

# Save animation
output_path = os.path.join(output_dir, "trefoil_torus.gif")
anim.save(output_path, writer="pillow", fps=30)

print(f"Animation saved to: {output_path}")
plt.close(fig)  # Close the figure to avoid displaying it in interactive environments

In [ ]:
from vpython import *
import numpy as np
import os
from PIL import Image

# Output path setup
output_dir = "output/trefoil_animation"
os.makedirs(output_dir, exist_ok=True)

# Parameters
r_tube = 0.1
num_steps = 200
step_size = 2 * np.pi / num_steps

points = []
for i in range(num_steps + 1):
    t = i * step_size
    x = (2 + np.cos(3 * t)) * np.cos(2 * t)
    y = (2 + np.cos(3 * t)) * np.sin(2 * t)
    z = np.sin(3 * t)
    points.append(vector(x, y, z))

scene.background = color.white
scene.width = 800
scene.height = 600
scene.camera.pos = vector(0, 0, -25)
scene.camera.axis = vector(0, 0, 25)

segments = []
for i in range(len(points) - 1):
    p1 = points[i]
    p2 = points[i + 1]
    segments.append(cylinder(pos=p1, axis=p2 - p1, radius=r_tube, color=vector(0, 0.25, 1)))

frames = []
num_frames = 120  # Reduced frame count for faster generation
rotation_per_frame = np.radians(3)  # 120 frames * 3° = full 360°

for frame in range(num_frames):
    rate(60)  # Controls rendering speed, you can lower to speed up even more
    for seg in segments:
        seg.rotate(angle=rotation_per_frame, axis=vector(0, 1, 0), origin=vector(0, 0, 0))

    # Capture frame
    filename = os.path.join(output_dir, f"frame_{frame:03}.png")
    scene.capture(filename)
    frames.append(filename)

# Create GIF
images = [Image.open(f) for f in frames]
gif_path = os.path.join(output_dir, "trefoil_vpython_fast.gif")
images[0].save(gif_path, save_all=True, append_images=images[1:], duration=33, loop=0)

print(f"GIF saved at: {gif_path}")
